# Benchmark 4 classifier (default vs tuned-F2), CXR vs CXR+EHR — 13 nhãn CheXpert (LP-FT fullmicro)

- Dataset **`fusion_dataset_lpft_fullmicro`** (embedding LP-FT + EHR fullmicro).
- Model: **XGBoost, LogReg, RidgeClassifier, LinearSVC** (SVM kernel bất khả thi ở 231k).
- Ngưỡng: **default** (0.5 proba / 0 decision) VÀ **tuned** (max-F2 trên val).
- Split **subject 70/10/20**. Train nhãn = **U-Ones**; eval = **nhãn thô loại -1, blank→0** (merge từ `manifest_cohort`).
- Heatmap 6 chỉ số: accuracy/precision/recall/f1/f2/auprc + **Δ (CXR+EHR − CXR)**.
- Chỉ ĐỌC fusion + manifest_cohort; output ghi **file mới** trong WRITE_DIR.

## 1. Load fusion + merge nhãn thô 4 trạng thái

In [ ]:
import os, numpy as np, pandas as pd
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)

WRITE_DIR='/content/drive/MyDrive/lpft_out'                      # nơi có fusion LP-FT (chỉnh nếu khác)
SRC_DIR  ='/content/drive/MyDrive/MIMIC_CXR/planB_out'           # nơi có manifest_cohort.parquet

df = pd.read_parquet(os.path.join(WRITE_DIR,'fusion_dataset_lpft_fullmicro.parquet'))
CHEX13=['Atelectasis','Cardiomegaly','Consolidation','Edema','Enlarged Cardiomediastinum',
 'Fracture','Lung Lesion','Lung Opacity','No Finding','Pleural Effusion',
 'Pleural Other','Pneumonia','Pneumothorax']                     # 13 (bỏ Support Devices)

# nhãn thô 1/0/-1/NaN từ manifest_cohort (fusion chỉ có y_ U-Ones -> cần raw cho eval sạch)
cm = pd.read_parquet(os.path.join(SRC_DIR,'manifest_cohort.parquet'))
cmr = cm[['dicom_id']+CHEX13].drop_duplicates('dicom_id').rename(columns={c:'raw_'+c for c in CHEX13})
df = df.merge(cmr, on='dicom_id', how='left')
RAW=['raw_'+c for c in CHEX13]
print('df:', df.shape, '| raw merged:', all(c in df.columns for c in RAW),
      '| thiếu raw:', int(df[RAW[0]].isna().all()))

## 2. Split subject 70/10/20 + feature + preprocess

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

rng=np.random.default_rng(42); uniq=np.unique(df.subject_id.values); rng.shuffle(uniq)
n=len(uniq); s1,s2=int(.70*n),int(.80*n)
trs,vas,tes=set(uniq[:s1]),set(uniq[s1:s2]),set(uniq[s2:])
sp=np.where(df.subject_id.isin(trs),'train',np.where(df.subject_id.isin(vas),'val','test'))
tr=sp=='train'; va=sp=='val'; te=sp=='test'
assert not (trs&tes) and not (trs&vas) and not (vas&tes)
print('subject 70/10/20 (ảnh):', {k:int((sp==k).sum()) for k in ['train','val','test']})

IMG=[f'img_{i}' for i in range(768)]
DROP={'dicom_id','subject_id','study_id','hadm_id','stay_id','first_hadm_id','first_admittime',
      'first_icd10_codes','study_dt','admittime','ed_intime','encounter_time','split','view',
      'disease_group','chiefcomplaint'}
if df['gender'].dtype=='object': df['gender']=df['gender'].map({'M':1,'F':0})
EHR=[c for c in df.columns if not c.startswith(('img_','y_','dx_','raw_')) and c not in DROP]
FEATSETS={'CXR':IMG, 'CXR+EHR':IMG+EHR}
print('IMG:',len(IMG),'| EHR:',len(EHR))

def prep(cols):
    X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy('float32')
    X=SimpleImputer(strategy='median').fit(X[tr]).transform(X)
    return StandardScaler().fit(X[tr]).transform(X).astype('float32')
Xcache={k:prep(v) for k,v in FEATSETS.items()}
print({k:v.shape for k,v in Xcache.items()})
RAWm=df[RAW].to_numpy('float32')     # 1/0/-1/NaN, khớp thứ tự dòng df

## 3. Train 4 model × 2 featset × 13 nhãn — default + tuned-F2
`SAMPLE_TRAIN` cap để chạy thử; `FRESH=True` làm lại sạch. Ridge/LinearSVC dùng `decision_function`.

In [ ]:
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,fbeta_score,
                             average_precision_score,roc_auc_score)
from tqdm.auto import tqdm
import warnings; warnings.filterwarnings('ignore')

SAMPLE_TRAIN=None    # None=full; đặt số (vd 80000) để chạy thử nhanh
FRESH=True
tr_idx=np.where(tr)[0]; va_idx=np.where(va)[0]; te_idx=np.where(te)[0]
if SAMPLE_TRAIN and len(tr_idx)>SAMPLE_TRAIN:
    tr_idx=np.random.default_rng(0).choice(tr_idx,SAMPLE_TRAIN,replace=False)

MODELS={
 'XGBoost':  lambda: XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.1,tree_method='hist',
                                   eval_metric='logloss',n_jobs=-1,random_state=0),
 'LogReg':   lambda: LogisticRegression(max_iter=1000,n_jobs=-1),
 'Ridge':    lambda: RidgeClassifier(),
 'LinearSVC':lambda: LinearSVC(C=1.0, dual=False, max_iter=2000),
}
def get_score(clf,X):
    if hasattr(clf,'predict_proba'): return clf.predict_proba(X)[:,1], 0.5   # proba
    return clf.decision_function(X), 0.0                                     # Ridge/LinearSVC

GRIDQ=np.linspace(0.01,0.99,99)
def best_thr(y,s):
    cand=np.unique(np.quantile(s,GRIDQ))
    f2=[fbeta_score(y,(s>=t).astype(int),beta=2,zero_division=0) for t in cand]
    return float(cand[int(np.argmax(f2))])
def metr(y,pred):
    return dict(accuracy=accuracy_score(y,pred),
                precision=precision_score(y,pred,zero_division=0),
                recall=recall_score(y,pred,zero_division=0),
                f1=f1_score(y,pred,zero_division=0),
                f2=fbeta_score(y,pred,beta=2,zero_division=0))

CKPT=os.path.join(WRITE_DIR,'benchmark_clf_lpft.csv')
if FRESH and os.path.exists(CKPT): os.remove(CKPT)
done=set(zip(pd.read_csv(CKPT).featset,pd.read_csv(CKPT).model)) if os.path.exists(CKPT) else set()

for fs,cols in FEATSETS.items():
    X=Xcache[fs]
    for mname,mk in MODELS.items():
        if (fs,mname) in done: print('bỏ qua',fs,mname); continue
        rows=[]
        for j,c in tqdm(list(enumerate(CHEX13)), desc=f'{fs} {mname}', leave=False):
            raw=RAWm[:,j]
            ytr=np.where(np.isnan(raw),0.,np.where(raw==-1,1.,raw)).astype(int)   # U-Ones train
            clf=mk().fit(X[tr_idx], ytr[tr_idx])
            sv,def_thr=get_score(clf,X[va_idx]); st,_=get_score(clf,X[te_idx])
            rt=raw[te_idx]; k=rt!=-1; yte=np.where(np.isnan(rt[k]),0.,rt[k]).astype(int); stk=st[k]
            rv=raw[va_idx]; kv=rv!=-1; yv=np.where(np.isnan(rv[kv]),0.,rv[kv]).astype(int); svk=sv[kv]
            auprc=average_precision_score(yte,stk) if yte.sum()>0 else np.nan
            auroc=roc_auc_score(yte,stk) if 0<yte.sum()<len(yte) else np.nan
            m_def=metr(yte,(stk>=def_thr).astype(int))
            tthr=best_thr(yv,svk); m_tun=metr(yte,(stk>=tthr).astype(int))
            for mode,mm,thr in [('default',m_def,def_thr),('tuned',m_tun,tthr)]:
                rows.append(dict(featset=fs,model=mname,label=c,thr_mode=mode,thr=round(float(thr),4),
                    **mm, auprc=auprc, auroc=auroc, prevalence=round(float(yte.mean()),4), n_pos=int(yte.sum())))
        pd.DataFrame(rows).to_csv(CKPT, mode='a', header=not os.path.exists(CKPT), index=False)
        print(f'✓ {fs} {mname} | auprc(mean)={np.mean([r["auprc"] for r in rows]):.3f}')

res=pd.read_csv(CKPT); print('\nrows:', len(res), '(kỳ vọng 2×4×13×2 =', 2*4*13*2, ')')

## 4. Heatmap — per model (CXR vs CXR+EHR) + Δ (cải thiện khi thêm EHR)

In [ ]:
import matplotlib.pyplot as plt, seaborn as sns
res=pd.read_csv(os.path.join(WRITE_DIR,'benchmark_clf_lpft.csv'))
METRICS=['accuracy','precision','recall','f1','f2','auprc']
MODELS_O=['XGBoost','LogReg','Ridge','LinearSVC']

def hm(sub, ax, title):
    m=sub.set_index('label')[METRICS].reindex(CHEX13)
    sns.heatmap(m,ax=ax,cmap='RdYlGn',vmin=0,vmax=1,annot=True,fmt='.3f',
                linewidths=.5,linecolor='black',annot_kws={'size':7},cbar_kws={'label':'score'})
    ax.set_title(title); ax.set_xlabel(''); ax.set_ylabel(''); ax.set_yticklabels(ax.get_yticklabels(),rotation=0)

# 4.1 per model — CXR vs CXR+EHR (tuned)
for mname in MODELS_O:
    fig,ax=plt.subplots(1,2,figsize=(16,6))
    for a,fs in zip(ax,['CXR','CXR+EHR']):
        hm(res[(res.model==mname)&(res.featset==fs)&(res.thr_mode=='tuned')], a, f'{mname} · {fs} · tuned-F2')
    plt.tight_layout(); plt.show()

# 4.2 Δ = (CXR+EHR − CXR), mỗi metric 1 heatmap (rows=nhãn, cols=model, tuned)
for metric in METRICS:
    D=pd.DataFrame(index=CHEX13)
    for mname in MODELS_O:
        a=res[(res.model==mname)&(res.featset=='CXR')     &(res.thr_mode=='tuned')].set_index('label')[metric].reindex(CHEX13)
        b=res[(res.model==mname)&(res.featset=='CXR+EHR')&(res.thr_mode=='tuned')].set_index('label')[metric].reindex(CHEX13)
        D[mname]=b-a
    plt.figure(figsize=(7,6))
    sns.heatmap(D,cmap='RdBu_r',center=0,vmin=-.15,vmax=.15,annot=True,fmt='+.3f',
                linewidths=.5,linecolor='black',annot_kws={'size':8},cbar_kws={'label':f'Δ {metric}'})
    plt.title(f'Δ (CXR+EHR − CXR) · {metric} · tuned  (đỏ=EHR giúp)'); plt.ylabel(''); plt.yticks(rotation=0)
    plt.tight_layout(); plt.show()

# 4.3 macro default vs tuned
macro=res.groupby(['model','featset','thr_mode'])[METRICS].mean().round(4)
print('== MACRO (mean 13 nhãn) =='); print(macro)